# CAM:macro_microphysics on its own

`macro_microphysics` is CAM5's cloud macrophysics + microphysics stage, one box
of the PI-atm workflow.  freeCAM can run it by itself: the cell below starts a
model, writes whatever it likes into the live StatePool, calls only that
process on all 512 ranks, and reports what the call produced.

Edit the input lines to feed it different inputs.  CAM physics carries dry
static energy and re-derives temperature from it at the stage's first internal
update, so a temperature perturbation has to be written into `s` as well --
otherwise the process silently discards it.  Any StatePool field listed by
`driver.cam.state.describe()` can be watched the same way, and any other
workflow process runs the same way: `driver.cam.workflow['deep_convection']`,
`['radiation']`, and so on.

The model stays live afterwards, so the call can be repeated against the state
the previous one left behind; `driver.close()` releases the MPI ranks.


In [ ]:
import freecam as fc

# CAM:macro_microphysics -- CAM5's cloud macrophysics + microphysics stage,
# called on its own.  One model: set whatever inputs you like, run only this
# process, and read back what it produced.
driver = fc.Driver(case='PI-atm', nsteps=1)
driver.initialize()
driver.run(steps=1)  # one complete step, so the stage starts from a real CAM state

state = driver.cam.state
macro_microphysics = driver.cam.workflow['macro_microphysics']

# ---- inputs: any StatePool field, any value ------------------------------
# CAM physics carries dry static energy and re-derives temperature from it at
# every physics update, so warm the atmosphere through both.
CPAIR = 1004.64  # J kg-1 K-1
state.T += 2.0                          # 2 K warmer everywhere
state['phys_state.s'] += 2.0 * CPAIR
state.q[:, :, 0, :] *= 1.05             # 5% more water vapour

# ---- run this one process, and nothing else ------------------------------
watched = {
    'temperature (K)': state.T,
    'dry static energy (J kg-1)': state['phys_state.s'],
    'water vapour (kg kg-1)': state.q[:, :, 0, :],
    'cloud liquid (kg kg-1)': state.q[:, :, 1, :],
    'cloud ice (kg kg-1)': state.q[:, :, 2, :],
    'droplet number (kg-1)': state.q[:, :, 3, :],
    'ice crystal number (kg-1)': state.q[:, :, 4, :],
    'heating rate (K s-1)': state['phys_tend.dtdt'],
}
before = {name: field.stats(rank='global') for name, field in watched.items()}
macro_microphysics.run()
after = {name: field.stats(rank='global') for name, field in watched.items()}

# ---- outputs: global means over all 512 ranks, padded columns excluded ----
display({
    name: {
        'before': before[name]['mean'],
        'after': after[name]['mean'],
        'change': after[name]['mean'] - before[name]['mean'],
    }
    for name in watched
})
